# LMA Phase 3: TELUGU Reasoning Finetuning

Finetunes the Phase 2 pretrained checkpoint on the synthetic comparative-reasoning QA set, using the real finetuning code imported from the same bundle used for pretraining (`kspsvln/lma-telugu-phase2`).


In [ ]:
# ===== CONFIG-ONLY CELL: Modify these before running =====
ROOT_DIR = "/kaggle/input/lma-telugu-phase2"  # Same code+config bundle as pretraining (dataset-metadata.json id: kspsvln/lma-telugu-phase2)
PRETRAINED_CKPT = "/kaggle/input/checkpoint-telugu/checkpoints/checkpoint_best.pt"  # Phase 2 pretrained checkpoint -- attach the checkpoint-telugu dataset as an input
OUT_DIR = "/kaggle/working/finetune_checkpoints"   # Save finetuned checkpoints here
CHECK_DIR = None                                    # Resume finetuning from a previous finetune session (if available)

LANGUAGE = "telugu"

# Hyperparameters: None means use config JSON defaults (configs/finetune_config.json)
# Must match finetune.py's actual argparse/run_finetuning fields.
hp = dict(
    batch_size=None,
    learning_rate=None,
    num_epochs=None,
    warmup_steps=None,
    weight_decay=None,
    amp=True,
)


In [ ]:
import sys
import os
import argparse
from pathlib import Path

# Fail fast if ROOT_DIR isn't actually where the bundle is mounted -- sys.path.insert()
# silently accepts a bad path, so a wrong ROOT_DIR would otherwise only surface later as a
# confusing "ModuleNotFoundError: No module named 'finetune'" at the import cell.
root_path = Path(ROOT_DIR)
expected_entry = root_path / "finetune" / "finetune.py"
if not expected_entry.exists():
    kaggle_input = Path("/kaggle/input")
    available = sorted(p.name for p in kaggle_input.iterdir()) if kaggle_input.exists() else []
    listing = "\n".join(f"  {p}" for p in sorted(root_path.iterdir())) if root_path.exists() else "  (ROOT_DIR does not exist)"
    raise RuntimeError(
        f"Expected {expected_entry} but it's not there.\n"
        f"ROOT_DIR = {ROOT_DIR}\n"
        f"Contents of ROOT_DIR:\n{listing}\n"
        f"Datasets attached under /kaggle/input/: {available}\n"
        "Check that the lma-telugu-phase2 bundle (with the finetune/ code and data already "
        "in it) is attached as an input to this notebook (Add Input) and that ROOT_DIR above "
        "matches its actual mounted path -- it may be nested one level deeper depending on "
        "how it was uploaded."
    )

if not Path(PRETRAINED_CKPT).exists():
    raise RuntimeError(
        f"PRETRAINED_CKPT={PRETRAINED_CKPT} does not exist. Attach the checkpoint-telugu "
        "dataset (kspsvln/checkpoint-telugu) as an input to this notebook, or update "
        "PRETRAINED_CKPT above to wherever your pretrained checkpoint is actually mounted."
    )

# Setup path to import from bundle
sys.path.insert(0, ROOT_DIR)
os.chdir('/kaggle/working')  # For checkpoint/log output

# Install tokenizers if needed
import subprocess
subprocess.run(['pip', 'install', 'tokenizers'], capture_output=True)

print(f'Root: {ROOT_DIR}')
print(f'Pretrained checkpoint: {PRETRAINED_CKPT}')
print(f'Output: {OUT_DIR}')
print(f'Resume: {CHECK_DIR}')


In [ ]:
# Import the real finetuning code (no reimplementation)
from finetune.finetune import run_finetuning

print('✅ Imported finetuning code from bundle')


In [ ]:
# Build command-line arguments by mimicking finetune.py's argparse
# Filter out None hyperparams (use config JSON defaults)
args_dict = {k: v for k, v in hp.items() if v is not None}

# Handle resume: compute resume_from path (resuming a FINETUNE session, separate from PRETRAINED_CKPT)
resume_from = None
if CHECK_DIR and Path(CHECK_DIR).exists():
    potential_ckpt = Path(CHECK_DIR) / 'checkpoint_last.pt'
    if potential_ckpt.exists():
        resume_from = str(potential_ckpt)
        print(f'Will resume finetuning from: {resume_from}')

# Create args namespace (fields must match finetune.py's run_finetuning() override_config exactly)
args = argparse.Namespace(
    batch_size=args_dict.get('batch_size', None),
    learning_rate=args_dict.get('learning_rate', None),
    num_epochs=args_dict.get('num_epochs', None),
    warmup_steps=args_dict.get('warmup_steps', None),
    weight_decay=args_dict.get('weight_decay', None),
    amp=args_dict.get('amp', True),
    pretrained_ckpt=PRETRAINED_CKPT,
    resume_from=resume_from,
)

print('✅ Arguments prepared')


In [ ]:
# Run finetuning with the real Trainer subclass (checkpointing, logging, AMP, masked QA loss, etc. all included)
# data_dir defaults to ROOT_DIR/finetune/data -- the QA dataset is bundled directly with the code
# since it's small (~10K short examples), no separate data input needed unlike pretraining's raw corpus.
print('\n' + '='*60)
print(f'Starting {LANGUAGE.upper()} reasoning finetuning...')
print('='*60 + '\n')

run_finetuning(args, root_dir=ROOT_DIR, data_dir=None, output_dir=OUT_DIR)

print('\n' + '='*60)
print('✅ Finetuning complete!')
print('='*60)


In [ ]:
# Final summary
import glob
import json

checkpoints = sorted(glob.glob(f'{OUT_DIR}/checkpoint_*.pt'))
logs = sorted(glob.glob(f'{OUT_DIR}/*.log'))
summary_path = Path(OUT_DIR) / 'finetune_summary.json'

print(f'\n📊 Finetuning outputs:')
if checkpoints:
    print(f'  Checkpoints:')
    for ckpt in checkpoints:
        size_mb = Path(ckpt).stat().st_size / 1e6
        print(f'    {ckpt} ({size_mb:.1f} MB)')

if logs:
    print(f'  Logs:')
    for log in logs:
        print(f'    {log}')

if summary_path.exists():
    summary = json.load(open(summary_path))
    print(f'\n📈 Pretrained vs. finetuned (PDF Sec 3.1 required comparison):')
    print(f'  Pretrained test exact-match accuracy: {summary["pretrained_test_accuracy"]}')
    print(f'  Finetuned test exact-match accuracy:  {summary["finetuned_test_accuracy"]}')
    print(f'  Best val_loss / val_ppl: {summary["best_val_loss"]:.4f} / {summary["best_val_ppl"]:.2f}')

print(f'\n📝 To resume finetuning in next run:')
print(f'  1. Save this notebook output as a Kaggle dataset')
print(f'  2. Set CHECK_DIR = "/kaggle/input/<this-output-dataset>/finetune_checkpoints"')
print(f'  3. Run the notebook again')
